<a href="https://colab.research.google.com/github/Dillybabu03/Brand-Reputation-Analysis-Indian-Brands/blob/main/Sentiment_Analysis_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install vaderSentiment
!pip install textblob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.3 MB/s eta 0:00:00


In [24]:
import os

# 1. Let's see the absolute root folders inside your mounted Google Drive
print("--- Root Drive Folders ---")
if os.path.exists('/content/drive/MyDrive'):
    print(os.listdir('/content/drive/MyDrive'))
else:
    print("Colab cannot access /content/drive/MyDrive at all.")

# 2. Let's try to search for any CSV files in your project directory dynamically
print("\n--- Searching for your raw CSV files ---")
possible_path = '/content/drive/MyDrive/SMRI_project'

if os.path.exists(possible_path):
    for root, dirs, files in os.walk(possible_path):
        for file in files:
            if file.endswith('.csv'):
                print(f"Found CSV at: {os.path.join(root, file)}")
else:
    print(f"The folder '{possible_path}' does not seem to exist. Check capital letters.")

--- Root Drive Folders ---
['SMRI_project']

--- Searching for your raw CSV files ---


In [25]:
import os

RAW_DATA_PATH = '/content/drive/MyDrive/SMRI_project/data/raw'

if os.path.exists(RAW_DATA_PATH):
    print(" Files found inside your raw folder:")
    for file in os.listdir(RAW_DATA_PATH):
        print(f" - {file}")
else:
    print(f" Path not found. Double-check your spelling: {RAW_DATA_PATH}")

 Path not found. Double-check your spelling: /content/drive/MyDrive/SMRI_project/data/raw


In [26]:
import os

# 1. Check if the drive is even mounted
if not os.path.exists('/content/drive'):
    print("❌ Your Google Drive is not connected. Click the 'Folder' icon on the left menu of Colab, then click the 'Mount Drive' button.")
else:
    print("📡 Drive connection open. Scanning for 'SMRI_project'...")
    found = False

    # 2. Search your entire drive layout dynamically
    for root, dirs, files in os.walk('/content/drive'):
        if 'SMRI_project' in dirs:
            exact_path = os.path.join(root, 'SMRI_project', 'data', 'raw')
            print(f"\n🎯 FOUND IT! Your exact RAW_DATA_PATH should be:")
            print(f"'{exact_path}'")

            # Show what files are inside if the path exists
            if os.path.exists(exact_path):
                print("\n📁 Files inside this folder:")
                print(os.listdir(exact_path))
            found = True
            break

    if not found:
        print("\n❌ Could not find a folder named 'SMRI_project' anywhere on your drive.")
        print("Please check your main Google Drive webpage to make sure the folder isn't inside a subfolder or another directory.")

📡 Drive connection open. Scanning for 'SMRI_project'...

🎯 FOUND IT! Your exact RAW_DATA_PATH should be:
'/content/drive/MyDrive/SMRI_project/data/raw'


In [30]:
import os
import shutil

print("🔄 Resetting stuck Drive mount points...")

# 1. Force unmount any active or broken connections
try:
    !団 = fusermount -u /content/drive
    from google.colab import drive
    drive.flush_and_unmount()
except:
    pass

# 2. Clean up the virtual directory if it got stuck with leftover files
stuck_path = '/content/drive'
if os.path.exists(stuck_path):
    try:
        shutil.rmtree(stuck_path)
        print("🧹 Cleared stuck mount directory.")
    except Exception as e:
        # If it's a system link, make a fresh sub-folder point instead
        !mkdir -p /content/drive_fresh

# 3. Mount fresh
from google.colab import drive
print("\n🔑 Requesting fresh Drive authorization link...")
drive.mount('/content/drive', force_remount=True)

🔄 Resetting stuck Drive mount points...
/bin/bash: line 1: 団: command not found

🔑 Requesting fresh Drive authorization link...
Mounted at /content/drive


In [31]:
import os

project_path = '/content/drive/MyDrive/SMRI_project'

if os.path.exists(project_path):
    print("📁 Found 'SMRI_project'. Let's see what's inside it:")
    print(os.listdir(project_path))

    # Let's check if there's a lowercase/uppercase mixup or if files are directly here
    for item in os.listdir(project_path):
        sub_path = os.path.join(project_path, item)
        if os.path.isdir(sub_path):
            print(f"   └── Subfolder found: '{item}' contains -> {os.listdir(sub_path)}")
else:
    print("❌ Cannot see 'SMRI_project'")

📁 Found 'SMRI_project'. Let's see what's inside it:
['data']
   └── Subfolder found: 'data' contains -> ['raw']


In [32]:
import pandas as pd
import os
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Set paths to point directly to your raw data
RAW_DATA_PATH = '/content/drive/MyDrive/SMRI_project/data/raw'
PROCESSED_DATA_PATH = '/content/drive/MyDrive/SMRI_project/data/processed'

# Automatically establish the processed folder structure if it doesn't exist
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# List of potential file variations to check for each brand
BRAND_FILES = {
    "zomato": ["zomato_raw_news.csv", "Zomato_raw_news.csv"],
    "ola_electric": ["ola_electric_raw_news.csv", "Ola_Electric_raw_news.csv"],
    "hdfc_bank": ["hdfc_bank_raw_news.csv", "HDFC_Bank_raw_news.csv"],
    "mamaearth": ["mamaearth_raw_news.csv", "Mamaearth_raw_news.csv"],
    "byjus": ["byju's_raw_news.csv", "byjus_raw_news.csv", "Byju's_raw_news.csv", "Byjus_raw_news.csv"]
}

analyzer = SentimentIntensityAnalyzer()
print(" Launching Sentiment Analysis Engine...")

for brand, variations in BRAND_FILES.items():
    raw_file = None

    # Check for files directly using system paths
    for filename in variations:
        target_path = os.path.join(RAW_DATA_PATH, filename)
        if os.path.exists(target_path):
            raw_file = target_path
            break

    if raw_file:
        print(f"\nProcessing sentiment metrics for: {brand}...")
        df = pd.read_csv(raw_file)
        df = df.dropna(subset=['text']).copy()

        vader_compounds = []
        vader_pos = []
        vader_neg = []
        vader_neu = []
        blob_polarities = []

        for idx, row in df.iterrows():
            headline = str(row['text'])

            # Compute VADER Metrics
            vader_score = analyzer.polarity_scores(headline)
            vader_compounds.append(vader_score['compound'])
            vader_pos.append(vader_score['pos'])
            vader_neg.append(vader_score['neg'])
            vader_neu.append(vader_score['neu'])

            # Compute TextBlob Polarity
            blob_score = TextBlob(headline)
            blob_polarities.append(blob_score.sentiment.polarity)

        # Append computed metrics back into the brand dataframe
        df['vader_compound'] = vader_compounds
        df['vader_pos'] = vader_pos
        df['vader_neg'] = vader_neg
        df['vader_neu'] = vader_neu
        df['textblob_polarity'] = blob_polarities

        # Save output to processed directory
        output_file = os.path.join(PROCESSED_DATA_PATH, f"{brand}_processed_sentiment.csv")
        df.to_csv(output_file, index=False)
        print(f" Successfully analyzed and saved {len(df)} rows to data/processed!")
    else:
        print(f" Warning: Could not locate raw file for {brand} inside {RAW_DATA_PATH}")

print("\n Analysis execution complete!")

 Launching Sentiment Analysis Engine...

Processing sentiment metrics for: zomato...
 Successfully analyzed and saved 100 rows to data/processed!

Processing sentiment metrics for: ola_electric...
 Successfully analyzed and saved 100 rows to data/processed!

Processing sentiment metrics for: hdfc_bank...
 Successfully analyzed and saved 100 rows to data/processed!

Processing sentiment metrics for: mamaearth...
 Successfully analyzed and saved 91 rows to data/processed!

Processing sentiment metrics for: byjus...
 Successfully analyzed and saved 100 rows to data/processed!

 Analysis execution complete!
